# distbench GPU correctness gate (Colab, single A100)

Run this **before** paying for a multi-GPU box. It confirms the CUDA/FSDP code works on a real GPU, so the Brev run just works. Set the runtime to an A100 GPU (Runtime -> Change runtime type -> A100), then run top to bottom.

This is a correctness gate, not a results run. One GPU cannot show multi-GPU scaling, the 1/N sharding number, real NCCL overhead, or the 8B model (8B OOMs on a single A100 even under FSDP). Those come from the Brev sweep, see docs/05_experiment_plan.md.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!git clone https://github.com/Saibernard/distributed_training.git
%cd distributed_training
!pip install -q -e ".[gpu]"

## The gate: single-GPU profiling + FSDP-on-CUDA
Runs the single-GPU profiler baseline and the FSDP path (the one that cannot init on a Mac), then asserts both produced valid results.

In [ ]:
# (1) single-GPU profiling baseline on the 1B model (real tokens/sec, peak mem, trace)
!python -m distbench.train --strategy single --model 1b --seq-len 2048 \
    --batch-size 1 --steps 20 --warmup 5 --dtype bf16 --profile \
    --out results/runs/single_1b_ws1.json

# (2) FSDP on CUDA at world_size 1 -- proves FSDP initializes and runs on a GPU
!torchrun --standalone --nproc_per_node=1 -m distbench.train \
    --strategy fsdp --model 1b --seq-len 2048 --batch-size 1 \
    --steps 10 --warmup 3 --dtype bf16 --activation-checkpointing \
    --out results/runs/fsdp_1b_ws1.json

In [ ]:
import json, os

def check(path, label):
    if not os.path.exists(path):
        return False, f"{label}: FAIL (no result file -- the run crashed)"
    d = json.load(open(path))
    if d.get('oom'):
        return False, f"{label}: FAIL (out of memory)"
    tps = d.get('tokens_per_sec_global')
    if not tps:
        return False, f"{label}: FAIL (no throughput recorded)"
    s = d.get('sharding', {})
    return True, (f"{label}: OK -- {d['strategy']} {d['model']} on {d['device']}, "
                  f"{tps:,.0f} tok/s, peak {d.get('peak_alloc_gb',0):.1f} GB, "
                  f"util {d.get('gpu_util_pct',0):.0f}%, shard_ratio {s.get('param_shard_ratio',1):.1f}")

ok1, m1 = check('results/runs/single_1b_ws1.json', 'single-GPU profiling')
ok2, m2 = check('results/runs/fsdp_1b_ws1.json',  'FSDP on CUDA      ')
print(m1)
print(m2)
passed = ok1 and ok2
print('\n' + '='*60)
print('GPU CORRECTNESS GATE:',
      'PASS -- safe to spend on the Brev multi-GPU sweep' if passed
      else 'FAIL -- fix this before renting a multi-GPU box')
print('='*60)
assert passed, 'gate failed: do not proceed to a paid multi-GPU run yet'

## The memory story (analytic, runs anywhere)
Why 8B needs FSDP: 128 GB of model state per GPU under DDP vs 16 GB under FSDP on 8 GPUs.

In [ ]:
!python -m distbench.memory --model 8b --world-size 8

## Plots from what ran here
Single-GPU figures only. The scaling, peak-memory, NCCL-overhead, and 1/N sharding figures need the multi-GPU Brev run.

In [ ]:
!python -m distbench.plot
from IPython.display import Image, display
import glob
for p in sorted(glob.glob('results/plots/*.png')):
    print(p); display(Image(p))